In [1]:
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
import pandas as pd
from config import RAW_DATA

df = pd.read_csv(RAW_DATA, sep="\t")

reviews = df["Review"]
labels = df["Liked"]

In [3]:
import re
from nltk.corpus import stopwords
from config import KEEP_NEGATIONS

stop_words = set(nltk.corpus.stopwords.words("english"))

if KEEP_NEGATIONS:
    negation_words = {
        "not",
        "no",
        "nor",
        "never",
        "none",
        "n't"
    }

    stop_words = stop_words - negation_words

In [4]:
import contractions
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words
    ]

    return " ".join(tokens)

preprocessed_reviews = reviews.apply(preprocess_text)

In [5]:
processed_df = pd.DataFrame({
    "Review": preprocessed_reviews,
    "Liked": labels
})

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    processed_df,
    test_size=0.2,
    random_state=42,
    stratify=processed_df["Liked"]
)

print(train_df.shape)
print(test_df.shape)

(800, 2)
(200, 2)


In [7]:
from config import TRAIN_CSV, TEST_CSV

train_df.to_csv(
    TRAIN_CSV,
    index=False
)

test_df.to_csv(
    TEST_CSV,
    index=False
)